# 🌀 NeuroForge CFD — AirfRANS training on a GPU

**Self-correcting, geometry-native AI CFD.** This notebook trains the NeuroForge
engine on the [AirfRANS](https://airfrans.readthedocs.io) dataset (incompressible
RANS over NACA airfoils) on a GPU, then runs the self-correcting solver
(predict → check physics residuals → estimate uncertainty → **Neural Residual
Iteration** → optional classical fallback) and visualises the results.

**Pipeline:** install → repo-local storage → download AirfRANS → train FNO +
corrector on GPU → evaluate field errors & Cl/Cd → self-correcting solve → plots.

All outputs land in the repo (`results/` is committed, `data/` + `checkpoints/`
are gitignored) — no Google Drive. Push results back with
`python scripts/push_results.py`.

> Runtime → *Change runtime type* → **GPU** (T4/L4/A100) before running.

## 0 · Check the GPU


In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU'


## 1 · Get the code & install

Set `REPO_URL` to your GitHub repo (push the local repo first), **or** upload the
project and point `REPO_DIR` at it.

In [ ]:
import os, sys, subprocess

# Use full CPU parallelism for data rasterisation on Colab. (The single-thread
# cap inside neuroforge.__init__ targets a different low-core host; we set the
# vars here BEFORE importing numpy/torch so this value wins.)
_n = str(os.cpu_count() or 4)
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_v] = _n

REPO_URL = 'https://github.com/ali-kin4/neuroforge-cfd.git'  # your repo
REPO_DIR = '/content/neuroforge-cfd'

if not os.path.isdir(REPO_DIR):
    rc = os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    if rc != 0:
        raise RuntimeError('Clone failed - set REPO_URL to your repo, or upload the project and set REPO_DIR.')

os.chdir(REPO_DIR)
# Editable install + the AirfRANS data extra. check=True surfaces any failure
# (a silent -q error was what caused 'No module named neuroforge').
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data]'], check=True)

# Guarantee importability in THIS kernel even if the editable path hook lags.
_src = os.path.join(REPO_DIR, 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

import neuroforge as nf
print('NeuroForge CFD', nf.__version__, 'ready at', REPO_DIR)


## 2 · Storage (repo-local — no Google Drive)

Everything lives in the repo. The big, regenerable artifacts (the AirfRANS
download + rasterised cache, and trained checkpoints) go in the **gitignored**
`data/` and `checkpoints/` dirs; the experiment **tables / figures** go in
`results/`, which **is** tracked so you can commit and push them back with
`python scripts/push_results.py`.

In [ ]:
# Repo-local storage — no Google Drive. Results live in the repo so you can
# git-commit and push them back (scripts/push_results.py).
DATA_ROOT   = os.path.join(REPO_DIR, 'data')            # raw AirfRANS  (gitignored, large)
CACHE_DIR   = os.path.join(REPO_DIR, 'data', 'cache')   # rasterised cache (gitignored, large)
CKPT_DIR    = os.path.join(REPO_DIR, 'checkpoints')     # trained .pt  (gitignored, large)
RESULTS_DIR = os.path.join(REPO_DIR, 'results')         # tables/figures (COMMITTED -> push back)
for d in (DATA_ROOT, CACHE_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)
print('repo-local storage (results/ is committed; data/ + checkpoints/ are gitignored):')
print('  DATA_ROOT   =', DATA_ROOT)
print('  CACHE_DIR   =', CACHE_DIR)
print('  CKPT_DIR    =', CKPT_DIR)
print('  RESULTS_DIR =', RESULTS_DIR)

## 3 - Imports

CPU thread counts were configured in the install cell (before numpy/torch
loaded). This cell adds a `sys.path` safety net so `import neuroforge` works
even if you run it on its own or after a kernel restart, then imports the API.


In [ ]:
import os, sys
for _p in ('/content/neuroforge-cfd/src', os.path.join(os.getcwd(), 'src')):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import torch, numpy as np, matplotlib.pyplot as plt
import neuroforge as nf

print('neuroforge', nf.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 4 - Dataset (auto-downloaded on first train)

You don't need to download anything by hand: the first training run fetches
AirfRANS to `DATA_ROOT` on a cache miss and then caches the rasterised tensors to
`CACHE_DIR` (`data/cache`). Later runs of the same config read the cache and skip
both the download and the rasterisation.

In [ ]:
TASK = 'full'   # 'full' = 800 train / 200 test. Use 'scarce' for a quicker pass.

# Preflight visibility: RAM, GPU, and whether the rasterised cache already exists.
import glob, torch
try:
    import psutil
    print(f'RAM: {psutil.virtual_memory().total/1e9:.0f} GB  (enable High-RAM for TASK=full)')
except Exception:
    pass
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (CPU only)')
_cached = glob.glob(os.path.join(CACHE_DIR, f'airfrans_{TASK}_*'))
print('cache present (fast run):', bool(_cached))
print('TASK =', TASK, '| dataset auto-downloads + caches on first train')


## 4b - Cache status (fast run or first run?)


In [ ]:
import glob
_hit = sorted(glob.glob(os.path.join(CACHE_DIR, f'airfrans_{TASK}_*')))
if _hit:
    print('CACHE HIT - training skips download + rasterisation:')
    for _p in _hit:
        print('   ', os.path.basename(_p))
else:
    print(f'No cache yet for TASK={TASK} - the first fit downloads + rasterises,')
    print('then caches ~300 MB to data/cache. Later runs of the SAME config are instant.')
print('Tip: keep resolution / n_train fixed so the cache is reused next session.')

## 5 - Train (clean NeuroForge API)

One object, scikit-learn style: hyper-parameters in the constructor, data in
`.fit()`. `corrector='deq'` uses the contractive Deep-Equilibrium corrector
(Banach convergence guarantee); `'local'` is the faster feed-forward one.
Checkpoints go to `checkpoints/` (gitignored); the dataset auto-downloads +
caches on first fit.

In [ ]:
# Corrector: 'deq' (contractive DEQ, Banach convergence guarantee) or 'local'.
CORRECTOR_TYPE = 'deq'

# One clean object: hyper-parameters here, data in .fit(...).
model = nf.NeuroForge(
    backbone='fno', width=48, n_layers=4, modes=20, dropout=0.05,
    corrector=CORRECTOR_TYPE, epochs=100, corrector_epochs=20,
    lr=8e-4, resolution=128, batch_size=6, amp=True, device='auto',
)
ckpt_path = os.path.join(CKPT_DIR, f'airfrans_{TASK}_fno.pt')

import traceback
try:
    model.fit('airfrans', task=TASK, n_train=300, n_val=100, root=DATA_ROOT,
              cache_dir=CACHE_DIR, download=True, out=ckpt_path)
    print(model)
    print('quick val rel-L2:', {k: round(v, 3) for k, v in model.metrics_.items()
                                if k in ('u', 'v', 'p', 'speed')})
except Exception:
    err = traceback.format_exc()
    print(err)
    with open(os.path.join(CKPT_DIR, 'last_error.txt'), 'w') as f:
        f.write(err)
    print('>>> error saved to', os.path.join(CKPT_DIR, 'last_error.txt'))


## 6 · Training curves & metrics


In [ ]:
h = model.history_['train']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(h['train_loss'], label='train')
ax[0].semilogy(h['val_loss'], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('composite loss')
ax[0].legend(); ax[0].set_title('backbone training')
hc = model.history_.get('corrector')
if hc:
    ax[1].semilogy(hc['corrector_loss'])
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('corrector loss')
    ax[1].set_title('corrector training')
plt.tight_layout(); plt.show()


## 6b - AirfRANS-protocol metrics (credible numbers)

Per-channel **MSE** (well-conditioned, unlike rel-L2), **surface** pressure MSE,
and the headline design metric **rho_Cl / rho_Cd** (Spearman rank correlation of
predicted-vs-true coefficients across the held-out set). It also runs the
make-or-break diagnostic: does a low PDE residual actually track low field error?


In [ ]:
M = model.evaluate(limit=60)
print('=== AirfRANS-protocol metrics (held-out) ===')
for k in ['mse_u', 'mse_v', 'mse_p', 'mse_nut', 'surface_mse_p', 'rho_cl', 'rho_cd',
          'cl_rel_err_mean', 'cd_rel_err_mean', 'residual_error_spearman',
          'residual_error_pearson']:
    if k in M:
        print('  %-24s %8.4f' % (k, M[k]))
print()
print('rho_cl / rho_cd: headline design-ranking metric (closer to 1 = better).')
print('residual_error_spearman > 0 means low residual tracks low error.')


## 6c - Corrector ablation + conformal calibration

**The make-or-break experiment:** does the corrector improve *accuracy* (field
MSE, rho_Cd) - not just the residual? And calibrate the trust map to a coverage
guarantee. (Set `CORRECTOR_TYPE='deq'` above for the principled contractive corrector.)


In [ ]:
abl = model.ablate_corrector(limit=60)
print('=== Corrector ablation: does correction improve ACCURACY? ===')
for k in ['mse_u', 'mse_p', 'rho_cl', 'rho_cd']:
    b = abl['backbone'].get(k, float('nan'))
    c = abl['corrected'].get(k, float('nan'))
    print('  %-8s backbone=%9.4f   corrected=%9.4f' % (k, b, c))
print('  (the corrector helps if corrected MSE < backbone MSE and rho closer to 1)')
print()
cal = model.calibrate(alpha=0.1, limit=40)
if cal is not None:
    print('Conformal calibration: q=%.3f  (90%% coverage band = q * sigma)' % cal.q)


## 6d - One-click ablation (the paper's main table)

Trains every arm (backbone | no-physics-loss | +local corrector | +DEQ
corrector) and writes a **mean+/-std** table to `results/ablation.md` + `.csv`.
This is heavier than a single fit, so the in-notebook defaults are a **fast**
version (1 seed, 40 epochs, 150 sims). For the **full paper run** (3 seeds, more
epochs) use the script at the bottom.


In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from benchmarks.ablation import run_ablation

try:
    ABL = run_ablation(
        'airfrans', task=TASK, n_train=150, n_val=80, resolution=128,
        seeds=(0,), epochs=40, corrector_epochs=10,
        width=48, modes=20, n_layers=4, batch_size=6,
        root=DATA_ROOT, cache_dir=CACHE_DIR, download=True,
        device='auto', out_dir=RESULTS_DIR, verbose=True,
    )
    print()
    print('Saved table ->', os.path.join(RESULTS_DIR, 'ablation.md'))
    print('FULL paper run (3 seeds, more epochs):')
    print('  !python scripts/run_full_research.py --preset full --cache-dir data/cache')
    print('Push results back: python scripts/push_results.py')
except Exception:
    import traceback
    traceback.print_exc()

## 7 · The self-correcting engine on a held-out airfoil

Load a few validation cases (with ground-truth fields), run the full
**predict → check → Neural Residual Iteration** loop, and inspect the residual
history (guaranteed non-increasing by the backtracking acceptance test) plus the
uncertainty/trust maps and the predicted Cl/Cd.


In [ ]:
from neuroforge.viz.plots import overview_figure
try:
    val = model.val_data(limit=12)
    case, gt = val[0]
    res = model.solve(case)
    print('case:', case.name)
    print('residual history:', [round(s['residual_norm'], 4) for s in res.history])
    print('Cl=%.4f  Cd=%.4f  trust_mean=%.3f' % (res.metrics['cl'], res.metrics['cd'], res.metrics['trust_mean']))
    overview_figure(res); plt.show()
except Exception as _e:
    print('[skipped]:', repr(_e))


## 8 · Prediction vs. ground truth

Side-by-side of predicted and CFD-reference fields on the held-out case, plus the
surface pressure coefficient (Cp).


In [ ]:
from neuroforge.viz.plots import plot_field, plot_cp
try:
    pred_field = res.field
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for col, key in enumerate(['speed', 'p', 'u']):
        plot_field(pred_field, key=key, ax=axes[0, col]); axes[0, col].set_title('pred ' + key)
        plot_field(gt, key=key, ax=axes[1, col]); axes[1, col].set_title('CFD ' + key)
    plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(figsize=(7, 4))
    plot_cp(pred_field, case, ax=ax, ref=gt); ax.set_title('Cp: prediction vs CFD'); plt.show()
except Exception as _e:
    print('[skipped]:', repr(_e))


## 9 · Aggregate Cl/Cd over the validation set


In [ ]:
from neuroforge.physics.metrics import force_coefficients
try:
    rows = []
    for case, gt in model.val_data(limit=40):
        pf = model.predict(case)
        fp, fg = force_coefficients(pf, case), force_coefficients(gt, case)
        rows.append((fp['cl'], fg['cl'], fp['cd'], fg['cd']))
    rows = np.array(rows)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    for i, (name, j) in enumerate([('Cl', 0), ('Cd', 2)]):
        ax[i].scatter(rows[:, j + 1], rows[:, j], s=18)
        lo, hi = rows[:, j:j + 2].min(), rows[:, j:j + 2].max()
        ax[i].plot([lo, hi], [lo, hi], 'k--', lw=1)
        ax[i].set_xlabel('CFD ' + name); ax[i].set_ylabel('pred ' + name); ax[i].set_title(name)
    plt.tight_layout(); plt.show()
except Exception as _e:
    print('[skipped]:', repr(_e))


## 10 · Save a report & next steps

- A standalone HTML report of the solve:


In [ ]:
try:
    report = res.save_report(os.path.join(RESULTS_DIR, 'sample_report.html'))
    print('report written:', report)
except Exception as _e:
    print('[skipped]:', repr(_e))

**Scale up for real accuracy:**

- `TASK='full'`, `n_train=800`, `n_val=200`.
- Bigger backbone: `ModelConfig(name='fno', width=64, modes=24, n_layers=5)`
  (or `name='transformer'` for the Transolver-style physics-attention model).
- More epochs (150–300) and a higher `resolution` (192/256) on an A100.
- The checkpoint in `checkpoints/` can be reloaded any time with
  `NeuroForgeEngine.from_checkpoint(ckpt_path)`.

See `docs/paper/neuroforge_cfd.md` for the method and `docs/ROADMAP.md` for what's next.

## One-shot evidence pack — ablation + OOD + calibration → results/

Runs the three reviewer-facing experiments in a single cell and writes every
table into the repo's `results/` dir, then assembles one `REPORT.md`:

* **Table 1** — in-distribution ablation, 3 seeds, mean±std (the deciding H1/H2/H3 result),
* **Table 3** — out-of-distribution ablation (`reynolds`, `aoa`) — the generalization gap,
* **H4** — trust-map calibration: empirical coverage + ECE per channel.

Re-uses the cached AirfRANS pairs, so it goes straight to training. Each stage is
independent — if one errors the others still finish and are saved. Push the
tables back with `python scripts/push_results.py`, then paste `REPORT.md` to
Claude to fold the real numbers into the paper.

In [ ]:
# === One-shot evidence pack: ablation + OOD + calibration -> results/ ===
# Budget: ~9 trainings (Table 1) + ~12 (OOD) + 1 (calibration). A couple of hours
# on an L4. Stages are independent: a failure in one is logged and the rest run.
import gc, os, sys, time, traceback
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from benchmarks.ablation import run_ablation, run_ood_ablation

RESULTS_DIR = os.path.join(REPO_DIR, 'results')   # repo-local; committed -> push back
os.makedirs(RESULTS_DIR, exist_ok=True)
SEEDS = (0, 1, 2)
report = {}

def _free():   # release memory between stages so ~20 trainings can't OOM
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

def _stage(name, fn):
    print(f"\n{'='*70}\n[{name}] starting ...\n{'='*70}")
    t0 = time.time()
    try:
        out = fn()
        print(f"[{name}] done in {(time.time()-t0)/60:.1f} min")
        return out
    except Exception:
        err = traceback.format_exc()
        print(err)
        report[name] = f"### {name} (FAILED)\n\n```\n{err}\n```\n"
        with open(os.path.join(RESULTS_DIR, f'{name}_error.txt'), 'w', encoding='utf-8') as f:
            f.write(err)
        return None
    finally:
        _free()

# --- 1) In-distribution ablation: paper Table 1 (mean +/- std over 3 seeds) ---
def _ablation():
    run_ablation('airfrans', task='full', n_train=400, n_val=120, resolution=128,
                 seeds=SEEDS, epochs=80, corrector_epochs=20,
                 width=48, modes=20, n_layers=4, batch_size=8,
                 root=DATA_ROOT, cache_dir=CACHE_DIR, download=True,
                 device='auto', out_dir=RESULTS_DIR, verbose=True)
    return open(os.path.join(RESULTS_DIR, 'ablation.md'), encoding='utf-8').read()
md = _stage('ablation', _ablation)
if md: report['ablation'] = md

# --- 2) Out-of-distribution ablation: paper Table 3 (reynolds + aoa) ---
def _ood():
    run_ood_ablation('airfrans', ood_tasks=('reynolds', 'aoa'),
                     n_train=400, n_val=120, resolution=128, seeds=SEEDS,
                     epochs=80, corrector_epochs=20,
                     width=48, modes=20, n_layers=4, batch_size=8,
                     root=DATA_ROOT, cache_dir=CACHE_DIR, download=True,
                     device='auto', out_dir=RESULTS_DIR, verbose=True)
    return open(os.path.join(RESULTS_DIR, 'ablation_ood.md'), encoding='utf-8').read()
md = _stage('ood', _ood)
if md: report['ood'] = md

# --- 3) Trust-map calibration quality: H4 (coverage + ECE per channel) ---
def _calibration():
    import neuroforge as nf
    from neuroforge.models.ensemble import MCDropoutUQ
    from neuroforge.physics.evaluation import evaluate_calibration
    g = globals().get('model', None)   # reuse the trained dropout model if present
    if (isinstance(g, nf.NeuroForge) and getattr(g, '_fitted', False)
            and float(g.config.model.dropout) > 0):
        m = g
        print('[calibration] reusing the model from the training cell.')
    else:
        print('[calibration] fitting a dedicated dropout model (dropout=0.05) ...')
        m = nf.NeuroForge(backbone='fno', width=48, n_layers=4, modes=20,
                          dropout=0.05, corrector='deq', epochs=80,
                          corrector_epochs=20, lr=8e-4, resolution=128,
                          batch_size=8, amp=True, device='auto').fit(
            'airfrans', task='full', n_train=400, n_val=120, root=DATA_ROOT,
            cache_dir=CACHE_DIR, download=True, verbose=False)
    dk = getattr(m, '_data_kw', {})
    base = MCDropoutUQ(m._model)
    class _UQ:
        def predict_with_uncertainty(self, x):
            return base.predict_with_uncertainty(x, n_samples=8)
    pairs = m.val_data()
    rows = []
    for ch, nm in [(0, 'u'), (1, 'v'), (2, 'p')]:
        r = evaluate_calibration(m.predictor.predict, _UQ(), pairs,
                                 alpha=0.1, channel=ch, n_bins=10)
        rows.append((nm, r))
        print(f"  channel {nm}: q={r['q']:.3f}  coverage={r['coverage']:.3f} "
              f"(target {r['target_coverage']:.2f})  ECE={r['ece']:.4f}")
    hdr = (f"### Calibration (H4) -- split-conformal trust map, alpha=0.10  "
           f"(task={dk.get('task','?')}, n_val={len(pairs)})")
    lines = [hdr, "",
             "| channel | q (multiplier) | empirical coverage | target | ECE |",
             "|---|---|---|---|---|"]
    for nm, r in rows:
        lines.append(f"| {nm} | {r['q']:.3f} | {r['coverage']:.3f} | "
                     f"{r['target_coverage']:.2f} | {r['ece']:.4f} |")
    lines += ["", "_Coverage in ~[0.85, 0.95] with a small ECE means the trust "
              "band is calibrated (H4 holds). Calibration set / test set = each "
              "half of the val split._"]
    txt = "\n".join(lines)
    with open(os.path.join(RESULTS_DIR, 'calibration.md'), 'w', encoding='utf-8') as f:
        f.write(txt + "\n")
    return txt
md = _stage('calibration', _calibration)
if md: report['calibration'] = md

# --- Assemble one REPORT.md (everything in one place, in results/) ---
titles = {'ablation': 'Table 1 -- In-distribution ablation (3 seeds)',
          'ood': 'Table 3 -- Out-of-distribution ablation',
          'calibration': 'H4 -- Trust-map calibration'}
parts = ["# NeuroForge -- evidence pack",
         f"_AirfRANS `full`, seeds {SEEDS}, FNO width=48 / modes=20, 80+20 epochs._", ""]
for k in ['ablation', 'ood', 'calibration']:
    parts += [f"## {titles[k]}", "", report.get(k, f"_({k} did not run)_"), ""]
report_path = os.path.join(RESULTS_DIR, 'REPORT.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("\n".join(parts) + "\n")

print(f"\n{'#'*70}\nALL DONE. Saved to results/ ({RESULTS_DIR}):")
for fn in ['ablation.md', 'ablation.csv', 'ablation_ood.md', 'ablation_ood.csv',
           'calibration.md', 'REPORT.md']:
    p = os.path.join(RESULTS_DIR, fn)
    print('   ', fn, '(OK)' if os.path.exists(p) else '(missing)')
print('#'*70)
print("\n>>> Push them back: python scripts/push_results.py")
print(">>> Then paste REPORT.md to Claude to fold the real numbers into the paper.")

### Render the evidence pack inline

Displays the assembled `REPORT.md` (Table 1 + Table 3 + H4 calibration) formatted
right here. Run after the evidence-pack cell. On Colab it also offers a one-click
download of every artifact (the committed copy lives in the repo's `results/`).

In [ ]:
# === Render REPORT.md (+ the individual tables) inline ===
import os
from IPython.display import Markdown, display

RESULTS_DIR = os.path.join(REPO_DIR, 'results')
report_path = os.path.join(RESULTS_DIR, 'REPORT.md')

if os.path.exists(report_path):
    display(Markdown(open(report_path, encoding='utf-8').read()))
else:
    # Fall back to whatever individual tables exist (e.g. mid-run).
    shown = False
    for fn in ['ablation.md', 'ablation_ood.md', 'calibration.md']:
        p = os.path.join(RESULTS_DIR, fn)
        if os.path.exists(p):
            display(Markdown(open(p, encoding='utf-8').read()))
            shown = True
    if not shown:
        print(f'No results yet in {RESULTS_DIR} -- run the evidence-pack cell first.')

# The artifacts are committed in results/ -> push them back with
#   python scripts/push_results.py
# On Colab you can also download them locally in one click:
try:
    from google.colab import files
    print('\nFiles in results/ (uncomment the loop below to download):')
    for fn in ['REPORT.md', 'ablation.md', 'ablation.csv', 'ablation_ood.md',
               'ablation_ood.csv', 'calibration.md']:
        p = os.path.join(RESULTS_DIR, fn)
        print('   ', p, '(OK)' if os.path.exists(p) else '(missing)')
    # for fn in ['REPORT.md', 'ablation.csv', 'ablation_ood.csv']:
    #     p = os.path.join(RESULTS_DIR, fn)
    #     if os.path.exists(p):
    #         files.download(p)
except Exception:
    pass  # not on Colab